In [10]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

# 第一步，加载需要微调的本地模型和分词器

In [11]:
# 1. 基础配置
model_path = "./model"  # 你之前下载的本地模型路径
output_dir = "./lora_results" # 微调后 LoRA 权重的保存位置

In [12]:
print("1. 正在加载模型与分词器...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
# 注意：微调时通常需要 pad_token。如果模型没有，我们指定 eos_token 为 pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 加载模型，这里我们为了节省内存，可以加载为 16 位浮点数 (bfloat16)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16, 
    device_map="auto" # 自动分配到 mps/cuda/cpu
)

1. 正在加载模型与分词器...


Loading weights: 100%|████████████████████████| 339/339 [00:03<00:00, 90.36it/s]


# 第二步：处理微调用到的数据

In [13]:
print("2. 正在加载并处理数据集...")
# 加载刚刚创建的 json 数据
dataset = load_dataset("json", data_files="new_dataset.json", split="train")
dataset

2. 正在加载并处理数据集...


Generating train split: 12 examples [00:00, 357.98 examples/s]


Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 12
})

In [14]:
# 定义一个将 instruction 和 output 拼接为对话模板的函数
def format_prompt(example):
    # 这里模拟模型的对话格式
    text = f"User: {example['instruction']}\nAssistant: {example['output']}{tokenizer.eos_token}"
    return {"text": text}

# 映射数据集
formatted_dataset = dataset.map(format_prompt)
formatted_dataset

Map: 100%|██████████████████████████████| 12/12 [00:00<00:00, 827.31 examples/s]


Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 12
})

# 第三步：配置 LoRA 参数，将普通模型包装成支持 LoRA 训练的模型

In [15]:
print("3. 正在配置 LoRA 核心参数...")
# 【关键学习点】这就是 LoRA 的魔法所在！我们不训练模型原本的上百亿参数，而是插入几个小矩阵。
lora_config = LoraConfig(
    r=8,               # Rank (秩)：决定了新增矩阵的大小。值越大，模型能学到的细节越多，但内存消耗也越大。通常 8, 16, 32 都可以。
    lora_alpha=16,     # 缩放系数：控制 LoRA 权重对原模型的影响力度，通常设为 r 的 2 倍。
    target_modules=["q_proj", "v_proj"], # 要在哪些层应用 LoRA？通常在注意力机制的 Query 和 Value 矩阵上效果最好。
    lora_dropout=0.05, # 防过拟合：随机丢弃 5% 的神经元。
    bias="none",
    task_type="CAUSAL_LM"
)

# 将普通模型包装成支持 LoRA 训练的模型
model = get_peft_model(model, lora_config)
# 打印一下看看我们省了多少参数：
model.print_trainable_parameters()

3. 正在配置 LoRA 核心参数...
trainable params: 1,089,536 || all params: 1,778,177,536 || trainable%: 0.0613


# 第四步：设置训练参数，开始模模型微调

In [17]:
print("4. 正在设置训练参数...")
training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2, # 每次处理的数据量，如果内存/显存爆了，就把这个调成 1
    gradient_accumulation_steps=2, # 梯度累加，变相增加 Batch Size，节省内存
    learning_rate=2e-4,          # 学习率  
    num_train_epochs=10,            # 训练轮数：把数据集看几遍
    logging_steps=1,               # 每隔多少步打印一次日志
    save_strategy="epoch",         # 保存策略
    optim="adamw_torch",
    
    dataset_text_field="text",       
)

4. 正在设置训练参数...


In [18]:
print("5. 初始化 SFTTrainer 并开始微调...")
# SFTTrainer 会帮我们自动处理文本的 Tokenize 和 Padding，非常省心
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
#     max_seq_length=512,  
    args=training_args,
)

# 开始训练！
trainer.train()

print("6. 训练完成，正在保存 LoRA 权重...")
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"LoRA 权重已成功保存至 {output_dir}")

5. 初始化 SFTTrainer 并开始微调...


Tokenizing train dataset: 100%|█████████| 12/12 [00:00<00:00, 475.76 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 151646, 'pad_token_id': 151643}.
/opt/miniconda3/envs/torch/lib/python3.10/site-packages/torch/utils/data/dataloader.py:774: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,3.696870
2,3.339038
3,3.512107
4,3.456466
5,3.510271
6,3.314931
7,3.356534
8,3.466073
9,3.151145
10,3.120412


/opt/miniconda3/envs/torch/lib/python3.10/site-packages/torch/utils/data/dataloader.py:774: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/miniconda3/envs/torch/lib/python3.10/site-packages/torch/utils/data/dataloader.py:774: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/miniconda3/envs/torch/lib/python3.10/site-packages/torch/utils/data/dataloader.py:774: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/miniconda3/envs/torch/lib/python3.10/site-packages/torch/utils/data/dataloader.py:774: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/miniconda3/envs/torch/lib/python3.10/site-packages/torc

6. 训练完成，正在保存 LoRA 权重...
LoRA 权重已成功保存至 ./lora_results
